### Imports

In [1]:
# For Production
import json
import random
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Colab only
from google.colab import userdata

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### Global variables

In [3]:
alphavantage_api = userdata.get('ALPHAVANTAGE_API')
ticker = 'NVDA'

### Functions

In [4]:
def convert_columns_to_numeric(df, columns_to_exclude=None):
    """
    Converts all columns in a DataFrame to numeric, except for those specified
    in the exclusion list.

    Args:
        df: pandas DataFrame.
        columns_to_exclude: A list of column names to exclude from conversion.
                            Defaults to None (convert all columns).

    Returns:
        pandas DataFrame with specified columns converted to numeric.
    """
    if columns_to_exclude is None:
        columns_to_exclude = []

    # Identify columns to convert to numeric
    columns_to_convert = df.columns.difference(columns_to_exclude)

    # Convert relevant columns to numeric, coercing errors
    for col in columns_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [5]:
def calculate_financial_growth(df, columns):
    """
    Calculates the Year-over-Year growth for specified columns in a DataFrame
    using a custom logic for negative previous values.

    Args:
        df: pandas DataFrame with financial statement data.
        columns: A list of column names for which to calculate YoY growth.

    Returns:
        pandas DataFrame with YoY growth columns added for the specified columns.
    """
    df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])
    df_sorted = df.sort_values(by='fiscalDateEnding').reset_index(drop=True)

    # Convert specified columns to numeric, coercing errors
    for col in columns:
        df_sorted[col] = pd.to_numeric(df_sorted[col], errors='coerce')

    # Define the custom Year-over-Year growth calculation function
    def calculate_yoy_growth_custom(current, previous):
        if pd.isna(previous) or previous == 0:
            return None
        if previous < 0:
            return ((current - previous) / abs(previous)) * 100
        else:
            return ((current - previous) / previous) * 100

    # Calculate Year-over-Year growth for each specified column using the custom function
    for col in columns:
        df_sorted[f'previous_{col}'] = df_sorted[col].shift(1)
        df_sorted[f'{col}_YoY_growth%'] = df_sorted.apply(
            lambda row: calculate_yoy_growth_custom(row[col], row[f'previous_{col}']),
            axis=1
        )
        df_sorted = df_sorted.drop(columns=[f'previous_{col}']) # Drop the temporary column

    return df_sorted[['fiscalDateEnding'] + [f'{col}_YoY_growth%' for col in columns]]

In [6]:
def impute_yoy_median(df):
    """
    Imputes the median for columns containing "YoY" in a DataFrame.

    Args:
        df: pandas DataFrame.

    Returns:
        pandas DataFrame with NaN values in YoY columns imputed with the median.
    """
    yoy_columns = [col for col in df.columns if 'YoY' in col]
    for col in yoy_columns:
        df[col] = df[col].fillna(df[col].median())
    return df

In [7]:
def convert_columns_to_millions(df, columns_to_convert):
    """
    Converts specified numeric columns in a DataFrame to millions.

    Args:
        df: pandas DataFrame.
        columns_to_convert: A list of column names to convert to millions.

    Returns:
        pandas DataFrame with specified columns converted to millions.
    """
    # Convert specified columns to numeric, coercing errors and then fillna
    for col in columns_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Divide the specified columns by 1,000,000
    for col in columns_to_convert:
        if col in df.columns:
            df[col] = df[col] / 1_000_000

    return df

In [8]:
def generate_financial_bar_charts(financial_data_transposed: pd.DataFrame):
    """
    Generates Plotly bar charts for financial metrics over time from a transposed DataFrame.

    Args:
        financial_data_transposed: A pandas DataFrame where the index are the metrics
                                   and columns are the years.

    Returns:
        A list of Plotly Figure objects.
    """
    figs = []
    # Convert the index to string type for Plotly compatibility
    financial_data_transposed.index = financial_data_transposed.index.astype(str)

    for index, row in financial_data_transposed.iterrows():
        # Determine the y-axis title based on the index name
        if index == 'EPS':
            yaxis_title = ''
        elif index.endswith('%'):
            yaxis_title = 'Porcentaje'
        else:
            yaxis_title = 'En millones'

        fig = go.Figure()
        # Generate a random color for the bars
        random_color = f'rgb({random.randint(0, 255)}, {random.randint(0, 255)}, {random.randint(0, 255)})'

        # Convert row values to numeric before plotting
        row_values_numeric = pd.to_numeric(row.values, errors='coerce')

        fig.add_trace(go.Bar(x=financial_data_transposed.columns, y=row_values_numeric, name=index, marker_color=random_color))
        fig.update_layout(
            title=index,  # Use the index name as the graph title
            xaxis_title='Year',
            yaxis_title=yaxis_title,
            xaxis=dict(
                tickmode='array',
                tickvals=financial_data_transposed.columns,
                ticktext=financial_data_transposed.columns
            )
        )
        figs.append(fig)

    return figs

## Data Ingestion

### Income Statement (Estado de resultados o P&L)

In [9]:
url_income = f'https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={ticker}&apikey={alphavantage_api}'
response_income = requests.get(url_income)
income_statement = response_income.json()

In [10]:
income_statement.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [11]:
# print(json.dumps(income_statement, indent=4))

In [12]:
income_df = pd.DataFrame.from_dict(income_statement['annualReports'])
income_df.shape

(20, 26)

In [13]:
income_df

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,None,2041000000,2300000000,259000000,None,None,None,2843000000,141450000000,21383000000,None,120067000000,None,141709000000,144552000000,120067000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,None,1539000000,1786000000,247000000,None,None,None,1864000000,84026000000,11146000000,None,72880000000,None,84273000000,86137000000,72880000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,None,609000000,866000000,257000000,None,None,None,1508000000,33818000000,4058000000,None,29760000000,None,34075000000,35583000000,29760000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,None,5000000,267000000,262000000,None,219000000,None,1543000000,4181000000,-187000000,None,4368000000,None,4443000000,5986000000,4368000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,None,-207000000,29000000,236000000,None,136000000,None,1174000000,9941000000,189000000,None,9752000000,None,10177000000,11351000000,9752000000
5,2021-01-31,USD,10396000000,16675000000,6279000000,6279000000,4532000000,1940000000,3924000000,5864000000,None,-127000000,57000000,184000000,None,61000000,None,1098000000,4409000000,77000000,None,4332000000,None,4593000000,5691000000,4332000000
6,2020-01-31,USD,6768000000,10918000000,4150000000,4150000000,2846000000,1093000000,2829000000,3922000000,None,126000000,178000000,52000000,None,176000000,None,381000000,2970000000,174000000,None,2796000000,None,3022000000,3403000000,2796000000
7,2019-01-31,USD,7171000000,11716000000,4545000000,4545000000,3804000000,991000000,2376000000,3367000000,None,78000000,136000000,58000000,None,150000000,None,262000000,3896000000,-245000000,None,4141000000,None,3954000000,4216000000,4141000000
8,2018-01-31,USD,5822000000,9714000000,3892000000,3892000000,3210000000,815000000,1797000000,2612000000,None,8000000,69000000,61000000,None,47000000,None,199000000,3196000000,149000000,None,3047000000,None,3257000000,3456000000,3047000000
9,2017-01-31,USD,4063000000,6910000000,2847000000,2847000000,1934000000,663000000,1463000000,2129000000,None,-4000000,54000000,58000000,None,29000000,None,187000000,1905000000,239000000,None,1666000000,None,1963000000,2150000000,1666000000


In [14]:
def convert_columns_to_numeric(df, columns_to_exclude=None):
    """
    Converts all columns in a DataFrame to numeric, except for those specified
    in the exclusion list.

    Args:
        df: pandas DataFrame.
        columns_to_exclude: A list of column names to exclude from conversion.
                            Defaults to None (convert all columns).

    Returns:
        pandas DataFrame with specified columns converted to numeric.
    """
    if columns_to_exclude is None:
        columns_to_exclude = []

    # Identify columns to convert to numeric
    columns_to_convert = df.columns.difference(columns_to_exclude)

    # Convert relevant columns to numeric, coercing errors
    for col in columns_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [15]:
income_df = convert_columns_to_numeric(income_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [16]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000


In [17]:
pd.DataFrame.from_dict(income_statement['quarterlyReports'])

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,51093000000,68127000000,17034000000,17034000000,44299000000,1282000000,5512000000,6794000000,None,495000000,568000000,73000000,None,None,None,812000000,50398000000,7438000000,None,42960000000,None,50471000000,51283000000,42960000000
1,2025-10-31,USD,41849000000,57006000000,15157000000,15157000000,36010000000,1134000000,4705000000,5839000000,None,563000000,624000000,61000000,None,None,None,751000000,37936000000,6026000000,None,31910000000,None,37997000000,38748000000,31910000000
2,2025-07-31,USD,33853000000,46743000000,12890000000,12890000000,28440000000,1122000000,4291000000,5413000000,None,530000000,592000000,62000000,None,None,None,669000000,31206000000,4784000000,None,26422000000,None,31268000000,31937000000,26422000000
3,2025-04-30,USD,26668000000,44062000000,17394000000,17394000000,21638000000,1041000000,3989000000,5030000000,None,452000000,515000000,63000000,None,None,None,611000000,21910000000,3135000000,None,18775000000,None,21973000000,22584000000,18775000000
4,2025-01-31,USD,28723000000,39331000000,10608000000,10608000000,24034000000,975000000,3714000000,4689000000,None,450000000,511000000,61000000,None,None,None,543000000,25217000000,3126000000,None,22091000000,None,25278000000,25821000000,22091000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,2007-01-31,USD,385706000,878873000,493167000,493167000,138514000,84900000,162276000,247192000,None,None,None,4000,None,None,None,34292000,151559000,-11947000,None,0,None,138514000,172806000,163506000
77,2006-10-31,USD,333942000,820572000,486630000,486630000,117613000,69100000,140732000,216329000,None,None,None,22000,None,None,None,25031000,128327000,21816000,None,0,None,117613000,142644000,106511000
78,2006-07-31,USD,292100000,687500000,395400000,395400000,95800000,69100000,127300000,196300000,None,None,None,6000,None,None,None,24200000,104500000,17800000,None,0,None,95800000,120000000,86800000
79,2006-04-30,USD,288757000,681807000,393050000,393050000,100685000,65668000,122404000,188072000,None,None,None,1000,None,None,None,24031000,109248000,18572000,None,0,None,100685000,124716000,90676000


### Balance Sheet (Hoja de balance)

In [18]:
url_balance = f'https://www.alphavantage.co/query?function=BALANCE_SHEET&symbol={ticker}&apikey={alphavantage_api}'
response_balance = requests.get(url_balance)
balance_sheet = response_balance.json()

In [19]:
balance_sheet.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [20]:
# print(json.dumps(balance_sheet, indent=4))

In [21]:
balance_df = pd.DataFrame.from_dict(balance_sheet['annualReports'])
balance_df.shape

(20, 38)

In [22]:
balance_df = convert_columns_to_numeric(balance_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [23]:
balance_df.tail()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding
15,2011-01-31,USD,4495246000,3226950000,665361000,665361000,345525000,348770000,1268296000,568857000.00,NaN,243171000,243171000,369844000,NaN,NaN,1825202000,2490563000,NaN,1313784000,942682000,286138000,NaN,NaN,336380000.00,371102000,23389000.00,23389000.00,NaN,NaN,359769000.00,71300000,347713000.00,3181462000,-1479392000.00,2149328000,677000,23547360000
16,2010-01-31,USD,3585918000,2480830000,447221000,447221000,330674000,374963000,1105088000,571858000.00,NaN,120458000,120458000,369844000,NaN,NaN,1281006000,1728227000,NaN,920778000,784378000,344527000,NaN,NaN,390277000.00,136400000,24450000.00,24450000.00,NaN,NaN,414727000.00,29950000,111950000.00,2665140000,-1463268000.00,1896182000,653000,21982960000
17,2009-01-31,USD,3350727000,2167958000,417688000,417688000,537834000,318435000,1182769000,625798000.00,NaN,147101000,147101000,369844000,NaN,NaN,837702000,56299000,NaN,956075000,778591000,218864000,NaN,NaN,NaN,177484000,NaN,25634000.00,NaN,NaN,25634000.00,559727000,NaN,2394652000,-1463268000.00,1964169000,629000,21925040000
18,2008-01-31,USD,3747671000,2888829000,726969000,726969000,358521000,666494000,858842000,359808000.00,NaN,106926000,106926000,354057000,NaN,NaN,1082509000,54336000,NaN,1129759000,967161000,492099000,NaN,NaN,NaN,162598000,NaN,NaN,NaN,NaN,NaN,469206000,NaN,2617912000,-1039632000.00,1994210000,619000,24269280000
19,2007-01-31,USD,2675263000,2031770000,544414000,544414000,354680000,518680000,643493000,260828000.00,NaN,45511000,45511000,301425000,NaN,NaN,573436000,40560000,NaN,668344000,638807000,272075000,NaN,NaN,NaN,29537000,NaN,NaN,NaN,NaN,NaN,365552000,NaN,2006919000,-487120000.00,1196565000,388000,23478500760


### Cashflow Statement (Estado de flujo de efectivo)

In [24]:
url_cashflow = f'https://www.alphavantage.co/query?function=CASH_FLOW&symbol={ticker}&apikey={alphavantage_api}'
response_cashflow = requests.get(url_cashflow)
cashflow_statement = response_cashflow.json()

In [25]:
cashflow_statement.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [26]:
# print(json.dumps(cashflow_statement, indent=4))

In [27]:
cashflow_df = pd.DataFrame.from_dict(cashflow_statement['annualReports'])
cashflow_df.shape

(20, 30)

In [28]:
cashflow_df = convert_columns_to_numeric(cashflow_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [29]:
cashflow_df.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome
0,2026-01-31,USD,102718000000,NaN,NaN,NaN,NaN,2843000000,6042000000,NaN,-11324000000,NaN,-52228000000,-48474000000,NaN,NaN,NaN,NaN,974000000.00,974000000.00,NaN,NaN,NaN,NaN,-40086000000,NaN,6386000000,NaN,NaN,120067000000
1,2025-01-31,USD,64089000000,NaN,NaN,NaN,NaN,1864000000,3236000000,NaN,-4781000000,NaN,-20421000000,-42359000000,NaN,NaN,NaN,NaN,834000000.00,834000000.00,NaN,NaN,NaN,NaN,-33706000000,NaN,4737000000,NaN,NaN,72880000000
2,2024-01-31,USD,28090000000,NaN,NaN,NaN,NaN,1508000000,1069000000,NaN,-98000000,NaN,-10566000000,-13633000000,NaN,NaN,NaN,NaN,395000000.00,395000000.00,NaN,NaN,NaN,NaN,-9533000000,NaN,3549000000,NaN,NaN,29760000000
3,2023-01-31,USD,5641000000,NaN,NaN,NaN,NaN,1544000000,1833000000,822000000.00,-2554000000,NaN,7375000000,-11617000000,NaN,NaN,NaN,NaN,398000000.00,398000000.00,NaN,NaN,NaN,NaN,-10039000000,NaN,2709000000,1399000000.00,NaN,4368000000
4,2022-01-31,USD,9108000000,NaN,NaN,NaN,NaN,1174000000,976000000,-2215000000.00,-774000000,NaN,-9830000000,1865000000,NaN,NaN,NaN,NaN,399000000.00,399000000.00,NaN,NaN,NaN,NaN,-1904000000,NaN,2004000000,1143000000.00,NaN,9752000000


### Earnings per share History (Ganancias por acción)

In [30]:
url_earnings = f'https://www.alphavantage.co/query?function=EARNINGS&symbol={ticker}&apikey={alphavantage_api}'
response_earnings = requests.get(url_earnings)
earnings_history = response_earnings.json()

In [31]:
earnings_history.keys()

dict_keys(['symbol', 'annualEarnings', 'quarterlyEarnings'])

In [32]:
# print(json.dumps(earnings_history, indent=4))

In [33]:
earnings_df = pd.DataFrame.from_dict(earnings_history['annualEarnings'])
earnings_df.shape

(27, 2)

In [34]:
earnings_df['reportedEPS'] = pd.to_numeric(earnings_df['reportedEPS'], errors='coerce')

In [35]:
earnings_df

,fiscalDateEnding,reportedEPS
0,2026-01-31,4.78
1,2025-01-31,2.99
2,2024-01-31,1.30
3,2023-01-31,0.33
4,2022-01-31,0.44
5,2021-01-31,0.25
6,2020-01-31,0.15
7,2019-01-31,0.17
8,2018-01-31,0.12
9,2017-01-31,0.06


## Feature Engineering

### Income Statement Feat. Eng.

In [36]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
earnings_df['fiscalDateEnding'] = pd.to_datetime(earnings_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, earnings_df, on='fiscalDateEnding', how='left')

In [37]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44


In [38]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
balance_df['fiscalDateEnding'] = pd.to_datetime(balance_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, balance_df[['fiscalDateEnding','commonStockSharesOutstanding']], on='fiscalDateEnding', how='left')

In [39]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000


In [40]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
cashflow_df['fiscalDateEnding'] = pd.to_datetime(cashflow_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, cashflow_df[['fiscalDateEnding','depreciationDepletionAndAmortization']], on='fiscalDateEnding', how='left')

In [41]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000,2843000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000,1864000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000,1508000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000,1544000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000,1174000000


In [42]:
# EBITDA Margin
income_df['ebitdaMargin%'] = income_df['ebitda']*100 / income_df['totalRevenue']
# EBIT Margin
income_df['ebitMargin%'] = income_df['operatingIncome']*100 / income_df['totalRevenue']
# Net income Margin
income_df['netIncomeMargin%'] = income_df['netIncome']*100 / income_df['totalRevenue']
# Net Interest
income_df['netInterest'] = income_df['interestIncome'] - income_df['interestExpense']
# Tax Rate
income_df['taxRate%'] = income_df['incomeTaxExpense']*100 / income_df['incomeBeforeTax']

In [43]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000,2843000000,66.94,60.38,55.60,2041000000.00,15.12
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000,1864000000,66.01,62.42,55.85,1539000000.00,13.26
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000,1508000000,58.41,54.12,48.85,609000000.00,12.00
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000,1544000000,22.19,15.66,16.19,5000000.00,-4.47
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000,1174000000,42.18,37.31,36.23,-207000000.00,1.90


In [44]:
income_df.tail()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%
15,2011-01-31,USD,1409090000,3543309000,2134219000,2134219000,255747000,362000000,848830000,1153343000,NaN,NaN,NaN,3127000,NaN,18549000.00,NaN,186989000,271169000,18023000,NaN,253146000.00,NaN,255747000,442736000,253146000,0.02,23547360000,186989000,12.49,7.22,7.14,NaN,6.65
16,2010-01-31,USD,1176923000,3326445000,2149522000,2149522000,-98945000,367017000,908851000,1275868000,NaN,NaN,NaN,3320000,NaN,19971000.00,NaN,196664000,-82294000,-14307000,NaN,-67987000.00,NaN,-78974000,117690000,-67987000,0.01,21982960000,196664000,3.54,-2.97,-2.04,NaN,17.39
17,2009-01-31,USD,1174269000,3424859000,2250590000,2250590000,-70700000,362000000,855879000,1244969000,NaN,NaN,NaN,406000,NaN,NaN,NaN,185023000,-42954000,-12913000,NaN,NaN,NaN,-42548000,142475000,-30041000,0.01,21925040000,185023000,4.16,-2.06,-0.88,NaN,30.06
18,2008-01-31,USD,1869280000,4097860000,2228580000,2228580000,836346000,341000000,691637000,1032934000,NaN,NaN,NaN,0,NaN,NaN,NaN,133192000,901341000,103696000,NaN,NaN,NaN,836346000,969538000,797645000,0.04,24269280000,133192000,23.66,20.41,19.46,NaN,11.50
19,2007-01-31,USD,1300449000,3068771000,1768322000,1768322000,453452000,294000000,553467000,846997000,NaN,NaN,NaN,21000,NaN,NaN,NaN,107562000,494480000,46350000,NaN,NaN,NaN,453452000,561014000,448834000,0.02,23478500760,107562000,18.28,14.78,14.63,NaN,9.37


In [45]:
income_df['reportedEPS'] = income_df['reportedEPS'].fillna(0)

In [46]:
income_df['interestIncome'] = income_df['interestIncome'].fillna(0)
income_df['netInterest'] = income_df['interestIncome'] - income_df['interestExpense']

In [47]:
income_columns_to_grow = ['totalRevenue','grossProfit','operatingIncome','ebitda',
                          'netIncome','reportedEPS','commonStockSharesOutstanding']
income_df_with_growth = calculate_financial_growth(income_df.copy(), income_columns_to_grow)

In [48]:
income_df_with_growth.head()

,fiscalDateEnding,totalRevenue_YoY_growth%,grossProfit_YoY_growth%,operatingIncome_YoY_growth%,ebitda_YoY_growth%,netIncome_YoY_growth%,reportedEPS_YoY_growth%,commonStockSharesOutstanding_YoY_growth%
0,2007-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008-01-31,33.53,43.74,84.44,72.82,77.71,57.26,3.37
2,2009-01-31,-16.42,-37.18,-108.45,-85.30,-103.77,-66.67,-9.66
3,2010-01-31,-2.87,0.23,-39.95,-17.40,-126.31,-15.38,0.26
4,2011-01-31,6.52,19.73,358.47,276.19,472.34,54.55,7.12


In [49]:
# Merge the two dataframes on 'fiscalDateEnding'
income_df = pd.merge(income_df, income_df_with_growth, on='fiscalDateEnding', how='left')

In [50]:
income_df.tail()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%,totalRevenue_YoY_growth%,grossProfit_YoY_growth%,operatingIncome_YoY_growth%,ebitda_YoY_growth%,netIncome_YoY_growth%,reportedEPS_YoY_growth%,commonStockSharesOutstanding_YoY_growth%
15,2011-01-31,USD,1409090000,3543309000,2134219000,2134219000,255747000,362000000,848830000,1153343000,NaN,NaN,0.00,3127000,NaN,18549000.00,NaN,186989000,271169000,18023000,NaN,253146000.00,NaN,255747000,442736000,253146000,0.02,23547360000,186989000,12.49,7.22,7.14,-3127000.00,6.65,6.52,19.73,358.47,276.19,472.34,54.55,7.12
16,2010-01-31,USD,1176923000,3326445000,2149522000,2149522000,-98945000,367017000,908851000,1275868000,NaN,NaN,0.00,3320000,NaN,19971000.00,NaN,196664000,-82294000,-14307000,NaN,-67987000.00,NaN,-78974000,117690000,-67987000,0.01,21982960000,196664000,3.54,-2.97,-2.04,-3320000.00,17.39,-2.87,0.23,-39.95,-17.40,-126.31,-15.38,0.26
17,2009-01-31,USD,1174269000,3424859000,2250590000,2250590000,-70700000,362000000,855879000,1244969000,NaN,NaN,0.00,406000,NaN,NaN,NaN,185023000,-42954000,-12913000,NaN,NaN,NaN,-42548000,142475000,-30041000,0.01,21925040000,185023000,4.16,-2.06,-0.88,-406000.00,30.06,-16.42,-37.18,-108.45,-85.30,-103.77,-66.67,-9.66
18,2008-01-31,USD,1869280000,4097860000,2228580000,2228580000,836346000,341000000,691637000,1032934000,NaN,NaN,0.00,0,NaN,NaN,NaN,133192000,901341000,103696000,NaN,NaN,NaN,836346000,969538000,797645000,0.04,24269280000,133192000,23.66,20.41,19.46,0.00,11.50,33.53,43.74,84.44,72.82,77.71,57.26,3.37
19,2007-01-31,USD,1300449000,3068771000,1768322000,1768322000,453452000,294000000,553467000,846997000,NaN,NaN,0.00,21000,NaN,NaN,NaN,107562000,494480000,46350000,NaN,NaN,NaN,453452000,561014000,448834000,0.02,23478500760,107562000,18.28,14.78,14.63,-21000.00,9.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
keep_income_cols = ['fiscalDateEnding','totalRevenue','totalRevenue_YoY_growth%',
                    'ebitda','ebitdaMargin%','ebitda_YoY_growth%','depreciationDepletionAndAmortization',
                    'operatingIncome','ebitMargin%','operatingIncome_YoY_growth%',
                    'interestExpense','interestIncome','netInterest',
                    'incomeBeforeTax','incomeTaxExpense','taxRate%',
                    'netIncome','netIncomeMargin%','netIncome_YoY_growth%',
                    'reportedEPS','reportedEPS_YoY_growth%',
                    'commonStockSharesOutstanding','commonStockSharesOutstanding_YoY_growth%']

In [52]:
income_df = income_df[keep_income_cols].copy()

In [53]:
income_df['reportedEPS_YoY_growth%'] = income_df['reportedEPS_YoY_growth%'].fillna(0)

In [54]:
income_df = impute_yoy_median(income_df)

In [55]:
income_df

,fiscalDateEnding,totalRevenue,totalRevenue_YoY_growth%,ebitda,ebitdaMargin%,ebitda_YoY_growth%,depreciationDepletionAndAmortization,operatingIncome,ebitMargin%,operatingIncome_YoY_growth%,interestExpense,interestIncome,netInterest,incomeBeforeTax,incomeTaxExpense,taxRate%,netIncome,netIncomeMargin%,netIncome_YoY_growth%,reportedEPS,reportedEPS_YoY_growth%,commonStockSharesOutstanding,commonStockSharesOutstanding_YoY_growth%
0,2026-01-31,215938000000,65.47,144552000000,66.94,67.82,2843000000,130387000000,60.38,60.08,259000000,2300000000.00,2041000000.00,141450000000,21383000000,15.12,120067000000,55.60,64.75,4.78,59.76,24432000000,-1.50
1,2025-01-31,130497000000,114.20,86137000000,66.01,142.07,1864000000,81453000000,62.42,147.04,247000000,1786000000.00,1539000000.00,84026000000,11146000000,13.26,72880000000,55.85,144.89,2.99,130.69,24804000000,-0.55
2,2024-01-31,60922000000,125.85,35583000000,58.41,494.44,1508000000,32972000000,54.12,680.59,257000000,866000000.00,609000000.00,33818000000,4058000000,12.00,29760000000,48.85,581.32,1.30,289.49,24940000000,-0.52
3,2023-01-31,26974000000,0.22,5986000000,22.19,-47.26,1544000000,4224000000,15.66,-57.93,262000000,267000000.00,5000000.00,4181000000,-187000000,-4.47,4368000000,16.19,-55.21,0.33,-25.08,25070000000,-1.10
4,2022-01-31,26914000000,61.40,11351000000,42.18,99.46,1174000000,10041000000,37.31,121.56,236000000,29000000.00,-207000000.00,9941000000,189000000,1.90,9752000000,36.23,125.12,0.44,77.45,25350000000,0.92
5,2021-01-31,16675000000,52.73,5691000000,34.13,67.23,1098000000,4532000000,27.18,59.24,184000000,57000000.00,-127000000.00,4409000000,77000000,1.75,4332000000,25.98,54.94,0.25,72.40,25120000000,1.62
6,2020-01-31,10918000000,-6.81,3403000000,31.17,-19.28,381000000,2846000000,26.07,-25.18,52000000,178000000.00,126000000.00,2970000000,174000000,5.86,2796000000,25.61,-32.48,0.15,-12.47,24720000000,-1.12
7,2019-01-31,11716000000,20.61,4216000000,35.98,21.99,262000000,3804000000,32.47,18.50,58000000,136000000.00,78000000.00,3896000000,-245000000,-6.29,4141000000,35.34,35.90,0.17,44.35,25000000000,-1.11
8,2018-01-31,9714000000,40.58,3456000000,35.58,60.74,199000000,3210000000,33.05,65.98,61000000,69000000.00,8000000.00,3196000000,149000000,4.66,3047000000,31.37,82.89,0.12,79.69,25280000000,-2.62
9,2017-01-31,6910000000,37.92,2150000000,31.11,117.83,187000000,1934000000,27.99,158.90,58000000,54000000.00,-4000000.00,1905000000,239000000,12.55,1666000000,24.11,171.34,0.06,113.33,25960000000,14.06


### FCF Feat. Eng.

In [56]:
cashflow_df.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome
0,2026-01-31,USD,102718000000,NaN,NaN,NaN,NaN,2843000000,6042000000,NaN,-11324000000,NaN,-52228000000,-48474000000,NaN,NaN,NaN,NaN,974000000.00,974000000.00,NaN,NaN,NaN,NaN,-40086000000,NaN,6386000000,NaN,NaN,120067000000
1,2025-01-31,USD,64089000000,NaN,NaN,NaN,NaN,1864000000,3236000000,NaN,-4781000000,NaN,-20421000000,-42359000000,NaN,NaN,NaN,NaN,834000000.00,834000000.00,NaN,NaN,NaN,NaN,-33706000000,NaN,4737000000,NaN,NaN,72880000000
2,2024-01-31,USD,28090000000,NaN,NaN,NaN,NaN,1508000000,1069000000,NaN,-98000000,NaN,-10566000000,-13633000000,NaN,NaN,NaN,NaN,395000000.00,395000000.00,NaN,NaN,NaN,NaN,-9533000000,NaN,3549000000,NaN,NaN,29760000000
3,2023-01-31,USD,5641000000,NaN,NaN,NaN,NaN,1544000000,1833000000,822000000.00,-2554000000,NaN,7375000000,-11617000000,NaN,NaN,NaN,NaN,398000000.00,398000000.00,NaN,NaN,NaN,NaN,-10039000000,NaN,2709000000,1399000000.00,NaN,4368000000
4,2022-01-31,USD,9108000000,NaN,NaN,NaN,NaN,1174000000,976000000,-2215000000.00,-774000000,NaN,-9830000000,1865000000,NaN,NaN,NaN,NaN,399000000.00,399000000.00,NaN,NaN,NaN,NaN,-1904000000,NaN,2004000000,1143000000.00,NaN,9752000000


In [57]:
fcf_df = cashflow_df[['fiscalDateEnding','operatingCashflow','depreciationDepletionAndAmortization','capitalExpenditures']].copy()

In [58]:
fcf_df['maintenanceCapex'] = fcf_df.apply(
    lambda row: row['capitalExpenditures'] if row['capitalExpenditures'] < row['depreciationDepletionAndAmortization'] else row['depreciationDepletionAndAmortization'],
    axis=1
)

In [59]:
fcf_df['growthCapex'] = fcf_df['capitalExpenditures'] - fcf_df['maintenanceCapex']

In [60]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0


In [61]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'])
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])

# Merge the dataframes
fcf_df = pd.merge(fcf_df, income_df[['fiscalDateEnding','totalRevenue','operatingIncome','interestExpense','incomeTaxExpense']], on='fiscalDateEnding', how='left')

In [62]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000


In [63]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'])
balance_df['fiscalDateEnding'] = pd.to_datetime(balance_df['fiscalDateEnding'])

# Merge the dataframes
fcf_df = pd.merge(fcf_df, balance_df[['fiscalDateEnding','inventory','currentNetReceivables',
                                      'currentAccountsPayable','deferredRevenue',
                                      'otherNonCurrentLiabilities','commonStockSharesOutstanding']], on='fiscalDateEnding', how='left')

In [64]:
fcf_df = fcf_df.fillna(0)

# Convert float columns to integers
float_cols = fcf_df.select_dtypes(include='float64').columns
for col in float_cols:
    fcf_df[col] = fcf_df[col].astype(int)

In [65]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,otherNonCurrentLiabilities,commonStockSharesOutstanding
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,381000000,24432000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,79000000,24804000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,65000000,24940000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,1913000000,25070000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,1553000000,25350000000


In [66]:
# Replace 'deferredRevenue' with 'otherNonCurrentLiabilities' if 'deferredRevenue' is not positive
fcf_df['deferredRevenue'] = fcf_df.apply(
    lambda row: row['otherNonCurrentLiabilities'] if row['deferredRevenue'] <= 0 else row['deferredRevenue'],
    axis=1
)

# Drop the 'otherNonCurrentLiabilities' column
fcf_df = fcf_df.drop(columns=['otherNonCurrentLiabilities'])

In [67]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,24432000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,24804000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,65000000,24940000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,25070000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,1553000000,25350000000


In [68]:
# Impute the median for 'deferredRevenue' if necessary
if fcf_df['deferredRevenue'].isnull().any():
    median_deferredRevenue = fcf_df['deferredRevenue'].median()
    fcf_df['deferredRevenue'] = fcf_df['deferredRevenue'].fillna(median_deferredRevenue)

In [69]:
fcf_df['ebitda'] = fcf_df['operatingIncome'] + fcf_df['depreciationDepletionAndAmortization']

In [70]:
fcf_df['workingCapital'] = fcf_df['inventory'] + fcf_df['currentNetReceivables'] - fcf_df['currentAccountsPayable'] - fcf_df['deferredRevenue']

In [71]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,24432000000,133230000000,49676000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,24804000000,83317000000,26756000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,65000000,24940000000,34480000000,12517000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,25070000000,5768000000,5880000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,1553000000,25350000000,11215000000,3919000000


In [72]:
fcf_df['previous_currentAccountsPayable'] = fcf_df['currentAccountsPayable'].shift(-1)
fcf_df['previous_workingCapital'] = fcf_df['workingCapital'].shift(-1)

fcf_df['changeInWC'] = fcf_df.apply(lambda row: (row['workingCapital'] - row['previous_workingCapital']) if row['previous_currentAccountsPayable'] > 0 else 0, axis=1)

# Drop the temporary columns
fcf_df = fcf_df.drop(columns=['previous_currentAccountsPayable', 'previous_workingCapital'])

In [73]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,24432000000,133230000000,49676000000,22920000000.00
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,24804000000,83317000000,26756000000,14239000000.00
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,65000000,24940000000,34480000000,12517000000,6637000000.00
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,25070000000,5768000000,5880000000,1961000000.00
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,1553000000,25350000000,11215000000,3919000000,2874000000.00


In [74]:
fcf_df['freeCashFlow'] = fcf_df['ebitda'] - fcf_df['interestExpense'] - fcf_df['incomeTaxExpense'] - fcf_df['maintenanceCapex'] - fcf_df['changeInWC']
fcf_df['freeCashFlowPerShare'] = fcf_df['freeCashFlow'] / fcf_df['commonStockSharesOutstanding']

In [75]:
fcf_df['freeCashFlow2'] = fcf_df['operatingCashflow'] - fcf_df['capitalExpenditures']

In [76]:
fcf_df['freeCashFlow'] - fcf_df['freeCashFlow2']

,0
0,-10851000000.00
1,-5032000000.00
2,-4562000000.00
3,-1620000000.00
4,-1192000000.00
5,-855000000.00
6,209000000.00
7,-163000000.00
8,-257000000.00
9,-789000000.00


In [77]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC,freeCashFlow,freeCashFlowPerShare,freeCashFlow2
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,24432000000,133230000000,49676000000,22920000000.00,85825000000.00,3.51,96676000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,24804000000,83317000000,26756000000,14239000000.00,55821000000.00,2.25,60853000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,65000000,24940000000,34480000000,12517000000,6637000000.00,22459000000.00,0.90,27021000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,25070000000,5768000000,5880000000,1961000000.00,2188000000.00,0.09,3808000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,1553000000,25350000000,11215000000,3919000000,2874000000.00,6940000000.00,0.27,8132000000


In [78]:
fcf_df_with_growth = calculate_financial_growth(fcf_df.copy(), ['freeCashFlow','freeCashFlowPerShare'])

In [79]:
fcf_df_with_growth.tail()

,fiscalDateEnding,freeCashFlow_YoY_growth%,freeCashFlowPerShare_YoY_growth%
15,2022-01-31,80.78,79.14
16,2023-01-31,-68.47,-68.12
17,2024-01-31,926.46,931.81
18,2025-01-31,148.55,149.91
19,2026-01-31,53.75,56.09


In [80]:
# Merge the two dataframes on 'fiscalDateEnding'
fcf_df = pd.merge(fcf_df, fcf_df_with_growth, on='fiscalDateEnding', how='left')

In [81]:
fcf_df.tail()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC,freeCashFlow,freeCashFlowPerShare,freeCashFlow2,freeCashFlow_YoY_growth%,freeCashFlowPerShare_YoY_growth%
15,2011-01-31,675797000,186989000,97890000,97890000,0,3543309000,255747000,3127000,18023000,345525000,348770000,286138000,347713000,23547360000,442736000,60444000,-188716000.00,512412000.00,0.02,577907000,22.19,14.07
16,2010-01-31,487807000,196664000,77601000,77601000,0,3326445000,-98945000,3320000,-14307000,330674000,374963000,344527000,111950000,21982960000,97719000,249160000,-388245000.00,419350000.00,0.02,410206000,357.77,357.09
17,2009-01-31,249360000,185023000,407670000,185023000,222647000,3424859000,-70700000,406000,-12913000,537834000,318435000,218864000,0,21925040000,114323000,637405000,104489000.00,-162682000.00,-0.01,-158310000,-120.31,-122.48
18,2008-01-31,1270196000,133192000,187745000,133192000,54553000,4097860000,836346000,0,103696000,358521000,666494000,492099000,0,24269280000,969538000,532916000,-68369000.00,801019000.00,0.03,1082451000,96.77,90.36
19,2007-01-31,587111000,107562000,145256000,107562000,37694000,3068771000,453452000,21000,46350000,354680000,518680000,272075000,0,23478500760,561014000,601285000,0.00,407081000.00,0.02,441855000,NaN,NaN


In [82]:
# Calculate the requested ratios and add them to the fcf_df DataFrame
fcf_df['maintenanceCapexMargin%'] = fcf_df['maintenanceCapex']*100 / fcf_df['totalRevenue']
fcf_df['workingCapitalMargin%'] = fcf_df['workingCapital']*100 / fcf_df['totalRevenue']
fcf_df['freeCashFlowMargin%'] = fcf_df['freeCashFlow']*100 / fcf_df['totalRevenue']
fcf_df['cashConversion%'] = fcf_df['freeCashFlow']*100 / fcf_df['ebitda']

In [83]:
keep_fcf_cols = ['fiscalDateEnding','ebitda','capitalExpenditures',
       'maintenanceCapex', 'growthCapex','interestExpense',
       'incomeTaxExpense','inventory','currentNetReceivables', 'currentAccountsPayable',
       'deferredRevenue','workingCapital','changeInWC', 'freeCashFlow',
       'freeCashFlow_YoY_growth%','freeCashFlowPerShare',
       'freeCashFlowPerShare_YoY_growth%','maintenanceCapexMargin%',
       'workingCapitalMargin%','freeCashFlowMargin%','cashConversion%']

In [84]:
fcf_df = fcf_df[keep_fcf_cols]

In [85]:
fcf_df.tail()

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
15,2011-01-31,442736000,97890000,97890000,0,3127000,18023000,345525000,348770000,286138000,347713000,60444000,-188716000.00,512412000.00,22.19,0.02,14.07,2.76,1.71,14.46,115.74
16,2010-01-31,97719000,77601000,77601000,0,3320000,-14307000,330674000,374963000,344527000,111950000,249160000,-388245000.00,419350000.00,357.77,0.02,357.09,2.33,7.49,12.61,429.14
17,2009-01-31,114323000,407670000,185023000,222647000,406000,-12913000,537834000,318435000,218864000,0,637405000,104489000.00,-162682000.00,-120.31,-0.01,-122.48,5.40,18.61,-4.75,-142.30
18,2008-01-31,969538000,187745000,133192000,54553000,0,103696000,358521000,666494000,492099000,0,532916000,-68369000.00,801019000.00,96.77,0.03,90.36,3.25,13.00,19.55,82.62
19,2007-01-31,561014000,145256000,107562000,37694000,21000,46350000,354680000,518680000,272075000,0,601285000,0.00,407081000.00,NaN,0.02,NaN,3.51,19.59,13.27,72.56


In [86]:
fcf_df = impute_yoy_median(fcf_df)

In [87]:
fcf_df

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
0,2026-01-31,133230000000,6042000000,2843000000,3199000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,49676000000,22920000000.00,85825000000.00,53.75,3.51,56.09,1.32,23.00,39.75,64.42
1,2025-01-31,83317000000,3236000000,1864000000,1372000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,26756000000,14239000000.00,55821000000.00,148.55,2.25,149.91,1.43,20.50,42.78,67.00
2,2024-01-31,34480000000,1069000000,1069000000,0,257000000,4058000000,5282000000,9999000000,2699000000,65000000,12517000000,6637000000.00,22459000000.00,926.46,0.90,931.81,1.75,20.55,36.87,65.14
3,2023-01-31,5768000000,1833000000,1544000000,289000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,5880000000,1961000000.00,2188000000.00,-68.47,0.09,-68.12,5.72,21.80,8.11,37.93
4,2022-01-31,11215000000,976000000,976000000,0,236000000,189000000,2605000000,4650000000,1783000000,1553000000,3919000000,2874000000.00,6940000000.00,80.78,0.27,79.14,3.63,14.56,25.79,61.88
5,2021-01-31,5630000000,1128000000,1098000000,30000000,184000000,77000000,1826000000,2429000000,1201000000,2009000000,1045000000,432000000.00,3839000000.00,-14.33,0.15,-15.69,6.58,6.27,23.02,68.19
6,2020-01-31,3227000000,489000000,381000000,108000000,52000000,174000000,979000000,1657000000,687000000,1336000000,613000000,-1861000000.00,4481000000.00,50.37,0.18,52.07,3.49,5.61,41.04,138.86
7,2019-01-31,4066000000,600000000,262000000,338000000,58000000,-245000000,1575000000,1424000000,511000000,14000000,2474000000,1011000000.00,2980000000.00,12.37,0.12,13.63,2.24,21.12,25.44,73.29
8,2018-01-31,3409000000,593000000,199000000,394000000,61000000,149000000,796000000,1265000000,596000000,2000000,1463000000,348000000.00,2652000000.00,275.11,0.10,285.20,2.05,15.06,27.30,77.79
9,2017-01-31,2121000000,176000000,176000000,0,58000000,239000000,794000000,826000000,485000000,20000000,1115000000,941000000.00,707000000.00,3.61,0.03,-9.16,2.55,16.14,10.23,33.33


### ROIC Feat. Eng.

In [88]:
roic_df = balance_df[['fiscalDateEnding', 'cashAndShortTermInvestments','shortTermInvestments','shortTermDebt',
                      'longTermDebt','capitalLeaseObligations','totalAssets','totalShareholderEquity']].copy()

In [89]:
roic_df.head()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalAssets,totalShareholderEquity
0,2026-01-31,10605000000,51951000000,1371000000.00,7469000000.00,2572000000.00,206803000000,157293000000
1,2025-01-31,8589000000,34621000000,288000000.00,8463000000.00,1807000000.00,111601000000,79327000000
2,2024-01-31,7280000000,18704000000,1478000000.00,8459000000.00,1347000000.00,65728000000,42978000000
3,2023-01-31,3389000000,9907000000,1250000000.00,9703000000.00,902000000.00,41182000000,22101000000
4,2022-01-31,1990000000,19218000000,144000000.00,10946000000.00,741000000.00,44187000000,26612000000


In [90]:
income_df.head()

,fiscalDateEnding,totalRevenue,totalRevenue_YoY_growth%,ebitda,ebitdaMargin%,ebitda_YoY_growth%,depreciationDepletionAndAmortization,operatingIncome,ebitMargin%,operatingIncome_YoY_growth%,interestExpense,interestIncome,netInterest,incomeBeforeTax,incomeTaxExpense,taxRate%,netIncome,netIncomeMargin%,netIncome_YoY_growth%,reportedEPS,reportedEPS_YoY_growth%,commonStockSharesOutstanding,commonStockSharesOutstanding_YoY_growth%
0,2026-01-31,215938000000,65.47,144552000000,66.94,67.82,2843000000,130387000000,60.38,60.08,259000000,2300000000.00,2041000000.00,141450000000,21383000000,15.12,120067000000,55.60,64.75,4.78,59.76,24432000000,-1.50
1,2025-01-31,130497000000,114.20,86137000000,66.01,142.07,1864000000,81453000000,62.42,147.04,247000000,1786000000.00,1539000000.00,84026000000,11146000000,13.26,72880000000,55.85,144.89,2.99,130.69,24804000000,-0.55
2,2024-01-31,60922000000,125.85,35583000000,58.41,494.44,1508000000,32972000000,54.12,680.59,257000000,866000000.00,609000000.00,33818000000,4058000000,12.00,29760000000,48.85,581.32,1.30,289.49,24940000000,-0.52
3,2023-01-31,26974000000,0.22,5986000000,22.19,-47.26,1544000000,4224000000,15.66,-57.93,262000000,267000000.00,5000000.00,4181000000,-187000000,-4.47,4368000000,16.19,-55.21,0.33,-25.08,25070000000,-1.10
4,2022-01-31,26914000000,61.40,11351000000,42.18,99.46,1174000000,10041000000,37.31,121.56,236000000,29000000.00,-207000000.00,9941000000,189000000,1.90,9752000000,36.23,125.12,0.44,77.45,25350000000,0.92


In [91]:
# Convert fiscalDateEnding to datetime for merging
roic_df['fiscalDateEnding'] = pd.to_datetime(roic_df['fiscalDateEnding'], errors='coerce')
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'], errors='coerce')

# Merge roic_df with income_df on fiscalDateEnding
roic_df = pd.merge(roic_df, income_df[['fiscalDateEnding','operatingIncome','taxRate%','netIncome']], on='fiscalDateEnding', how='left')

In [92]:
fcf_df.head()

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
0,2026-01-31,133230000000,6042000000,2843000000,3199000000,259000000,21383000000,21403000000,38466000000,9812000000,381000000,49676000000,22920000000.00,85825000000.00,53.75,3.51,56.09,1.32,23.00,39.75,64.42
1,2025-01-31,83317000000,3236000000,1864000000,1372000000,247000000,11146000000,10080000000,23065000000,6310000000,79000000,26756000000,14239000000.00,55821000000.00,148.55,2.25,149.91,1.43,20.50,42.78,67.00
2,2024-01-31,34480000000,1069000000,1069000000,0,257000000,4058000000,5282000000,9999000000,2699000000,65000000,12517000000,6637000000.00,22459000000.00,926.46,0.90,931.81,1.75,20.55,36.87,65.14
3,2023-01-31,5768000000,1833000000,1544000000,289000000,262000000,-187000000,5159000000,3827000000,1193000000,1913000000,5880000000,1961000000.00,2188000000.00,-68.47,0.09,-68.12,5.72,21.80,8.11,37.93
4,2022-01-31,11215000000,976000000,976000000,0,236000000,189000000,2605000000,4650000000,1783000000,1553000000,3919000000,2874000000.00,6940000000.00,80.78,0.27,79.14,3.63,14.56,25.79,61.88


In [93]:
# Convert fiscalDateEnding to datetime for merging
roic_df['fiscalDateEnding'] = pd.to_datetime(roic_df['fiscalDateEnding'], errors='coerce')
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'], errors='coerce')

# Merge roic_df with fcf_df on fiscalDateEnding
roic_df = pd.merge(roic_df, fcf_df[['fiscalDateEnding','growthCapex','freeCashFlow']], on='fiscalDateEnding', how='left')

In [94]:
# Identify columns to convert to numeric (all except 'fiscalDateEnding')
roic_df = convert_columns_to_numeric(roic_df, columns_to_exclude=['fiscalDateEnding'])

In [95]:
roic_df.head()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalAssets,totalShareholderEquity,operatingIncome,taxRate%,netIncome,growthCapex,freeCashFlow
0,2026-01-31,10605000000,51951000000,1371000000.00,7469000000.00,2572000000.00,206803000000,157293000000,130387000000,15.12,120067000000,3199000000,85825000000.00
1,2025-01-31,8589000000,34621000000,288000000.00,8463000000.00,1807000000.00,111601000000,79327000000,81453000000,13.26,72880000000,1372000000,55821000000.00
2,2024-01-31,7280000000,18704000000,1478000000.00,8459000000.00,1347000000.00,65728000000,42978000000,32972000000,12.00,29760000000,0,22459000000.00
3,2023-01-31,3389000000,9907000000,1250000000.00,9703000000.00,902000000.00,41182000000,22101000000,4224000000,-4.47,4368000000,289000000,2188000000.00
4,2022-01-31,1990000000,19218000000,144000000.00,10946000000.00,741000000.00,44187000000,26612000000,10041000000,1.90,9752000000,0,6940000000.00


In [96]:
roic_df.describe()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalAssets,totalShareholderEquity,operatingIncome,taxRate%,netIncome,growthCapex,freeCashFlow
count,20,20.00,20.00,17.00,18.00,15.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00
mean,2016-08-01 00:00:00,2829627800.00,9151928000.00,578274588.24,3439261833.33,586057466.67,30081163000.00,20758294600.00,14003871900.00,10.31,12851961150.00,303044700.00,9637868300.00
min,2007-01-31 00:00:00,417688000.00,1000000.00,15000000.00,18998000.00,6000000.00,2675263000.00,2006919000.00,-98945000.00,-6.29,-67987000.00,0.00,-162682000.00
25%,2011-10-31 18:00:00,648020750.00,1689153000.00,144000000.00,40975500.00,18249000.00,5288507500.00,3904658500.00,610179250.00,5.56,534110500.00,0.00,523523250.00
50%,2016-08-01 00:00:00,814500000.00,3823454000.00,311030000.00,1984000000.00,24450000.00,8605500000.00,5294851500.00,1385173000.00,12.20,1231822500.00,23000000.00,802138000.00
75%,2021-05-02 06:00:00,3542250000.00,10108750000.00,999000000.00,7092750000.00,821500000.00,31888750000.00,18195000000.00,4301000000.00,15.05,4341000000.00,239235250.00,3999500000.00
max,2026-01-31 00:00:00,10896000000.00,51951000000.00,1500000000.00,10946000000.00,2572000000.00,206803000000.00,157293000000.00,130387000000.00,30.06,120067000000.00,3199000000.00,85825000000.00
std,NaN,3541368449.71,13188673482.99,533556733.54,3874322113.52,789902469.31,49692828941.78,37136852597.99,33229163290.73,8.31,30333480687.20,750405250.17,22047556660.48


In [97]:
# Tax Rate is 21% in USA
roic_df['taxRate%'] = roic_df['taxRate%'].fillna(21.0)

In [98]:
# Fill NaN values with 0 before performing the calculation
roic_df = roic_df.fillna(0)

In [99]:
# Create the 'investedCapital' feature
roic_df['investedCapital'] = roic_df['totalShareholderEquity'] + roic_df['shortTermDebt'] + roic_df['longTermDebt'] + roic_df['capitalLeaseObligations'] - roic_df['shortTermInvestments']

In [100]:
# Calculate ROA, ROE, NOPAT, and ROIC
roic_df['ROA%'] = roic_df['netIncome']*100 / roic_df['totalAssets']
roic_df['ROE%'] = roic_df['netIncome']*100 / roic_df['totalShareholderEquity']
roic_df['NOPAT'] = roic_df['operatingIncome'] * (1 - roic_df['taxRate%'] / 100)
roic_df['ROIC%'] = roic_df['NOPAT']*100 / roic_df['investedCapital']

In [101]:
# Replace infinite values with 0 in the specified columns
for col in ['ROA%', 'ROE%', 'ROIC%']:
    roic_df[col] = roic_df[col].replace([np.inf, -np.inf], 0)

In [102]:
roic_df.head()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalAssets,totalShareholderEquity,operatingIncome,taxRate%,netIncome,growthCapex,freeCashFlow,investedCapital,ROA%,ROE%,NOPAT,ROIC%
0,2026-01-31,10605000000,51951000000,1371000000.00,7469000000.00,2572000000.00,206803000000,157293000000,130387000000,15.12,120067000000,3199000000,85825000000.00,116754000000.00,58.06,76.33,110676393983.74,94.79
1,2025-01-31,8589000000,34621000000,288000000.00,8463000000.00,1807000000.00,111601000000,79327000000,81453000000,13.26,72880000000,1372000000,55821000000.00,55264000000.00,65.30,91.87,70648306952.61,127.84
2,2024-01-31,7280000000,18704000000,1478000000.00,8459000000.00,1347000000.00,65728000000,42978000000,32972000000,12.00,29760000000,0,22459000000.00,35558000000.00,45.28,69.24,29015515997.40,81.60
3,2023-01-31,3389000000,9907000000,1250000000.00,9703000000.00,902000000.00,41182000000,22101000000,4224000000,-4.47,4368000000,289000000,2188000000.00,24049000000.00,10.61,19.76,4412923224.11,18.35
4,2022-01-31,1990000000,19218000000,144000000.00,10946000000.00,741000000.00,44187000000,26612000000,10041000000,1.90,9752000000,0,6940000000.00,19225000000.00,22.07,36.65,9850098782.82,51.24


In [103]:
roic_df.describe()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalAssets,totalShareholderEquity,operatingIncome,taxRate%,netIncome,growthCapex,freeCashFlow,investedCapital,ROA%,ROE%,NOPAT,ROIC%
count,20,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00
mean,2016-08-01 00:00:00,2829627800.00,9151928000.00,491533400.00,3095335650.00,439543100.00,30081163000.00,20758294600.00,14003871900.00,10.31,12851961150.00,303044700.00,9637868300.00,15632778750.00,19.55,28.85,12229358023.35,41.48
min,2007-01-31 00:00:00,417688000.00,1000000.00,0.00,0.00,0.00,2675263000.00,2006919000.00,-98945000.00,-6.29,-67987000.00,0.00,-162682000.00,1433483000.00,-1.90,-2.55,-81743185.59,-4.48
25%,2011-10-31 18:00:00,648020750.00,1689153000.00,91000000.00,24184750.00,4500000.00,5288507500.00,3904658500.00,610179250.00,5.56,534110500.00,0.00,523523250.00,1802337750.00,8.65,13.22,520087974.31,18.25
50%,2016-08-01 00:00:00,814500000.00,3823454000.00,298454000.00,1690714000.00,20218500.00,8605500000.00,5294851500.00,1385173000.00,12.20,1231822500.00,23000000.00,802138000.00,3082080000.00,15.60,22.64,1215744558.94,33.34
75%,2021-05-02 06:00:00,3542250000.00,10108750000.00,870000000.00,6340250000.00,674250000.00,31888750000.00,18195000000.00,4301000000.00,15.05,4341000000.00,239235250.00,3999500000.00,16009000000.00,23.33,37.68,4422905391.55,48.96
max,2026-01-31 00:00:00,10896000000.00,51951000000.00,1500000000.00,10946000000.00,2572000000.00,206803000000.00,157293000000.00,130387000000.00,30.06,120067000000.00,3199000000.00,85825000000.00,116754000000.00,65.30,91.87,110676393983.74,127.84
std,NaN,3541368449.71,13188673482.99,533491401.30,3814568064.07,726318414.04,49692828941.78,37136852597.99,33229163290.73,8.31,30333480687.20,750405250.17,22047556660.48,27657764499.66,18.15,25.26,28345293500.28,33.51


In [104]:
# Calculate reinvestmentRate based on the condition
roic_df['reinvestmentRate%'] = roic_df.apply(lambda row: row['growthCapex']*100/row['freeCashFlow'] if row['freeCashFlow'] > 0 else 0, axis=1)

In [105]:
keep_roic_cols = ['fiscalDateEnding','cashAndShortTermInvestments','shortTermInvestments',
                  'shortTermDebt','longTermDebt','capitalLeaseObligations',
                  'totalShareholderEquity','investedCapital','ROA%','ROE%','ROIC%','reinvestmentRate%']

In [106]:
roic_df = roic_df[keep_roic_cols].copy()

In [107]:
roic_df

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalShareholderEquity,investedCapital,ROA%,ROE%,ROIC%,reinvestmentRate%
0,2026-01-31,10605000000,51951000000,1371000000.00,7469000000.00,2572000000.00,157293000000,116754000000.00,58.06,76.33,94.79,3.73
1,2025-01-31,8589000000,34621000000,288000000.00,8463000000.00,1807000000.00,79327000000,55264000000.00,65.30,91.87,127.84,2.46
2,2024-01-31,7280000000,18704000000,1478000000.00,8459000000.00,1347000000.00,42978000000,35558000000.00,45.28,69.24,81.60,0.00
3,2023-01-31,3389000000,9907000000,1250000000.00,9703000000.00,902000000.00,22101000000,24049000000.00,10.61,19.76,18.35,13.21
4,2022-01-31,1990000000,19218000000,144000000.00,10946000000.00,741000000.00,26612000000,19225000000.00,22.07,36.65,51.24,0.00
5,2021-01-31,847000000,10714000000,999000000.00,5964000000.00,634000000.00,16893000000,13776000000.00,15.05,25.64,32.32,0.78
6,2020-01-31,10896000000,1000000,91000000.00,1991000000.00,652000000.00,12204000000,14937000000.00,16.15,22.91,17.94,2.41
7,2019-01-31,782000000,6640000000,91000000.00,1988000000.00,0.00,9342000000,4781000000.00,31.15,44.33,84.57,11.34
8,2018-01-31,4002000000,3106000000,15000000.00,1985000000.00,0.00,7471000000,6365000000.00,27.11,40.78,48.08,14.86
9,2017-01-31,1766000000,5032000000,827000000.00,1983000000.00,6000000.00,5762000000,3546000000.00,16.93,28.91,47.70,0.00


### KPIs Feat. Eng.

In [122]:
kpis_df = cashflow_df[['fiscalDateEnding','dividendPayoutCommonStock','dividendPayoutPreferredStock',
                       'paymentsForRepurchaseOfCommonStock','paymentsForRepurchaseOfPreferredStock',]].copy()

In [123]:
# Convert fiscalDateEnding to datetime for merging
kpis_df['fiscalDateEnding'] = pd.to_datetime(kpis_df['fiscalDateEnding'], errors='coerce')
balance_df['fiscalDateEnding'] = pd.to_datetime(balance_df['fiscalDateEnding'], errors='coerce')

# Merge kpis_df with balance_df on fiscalDateEnding
kpis_df = pd.merge(kpis_df, balance_df[['fiscalDateEnding','shortTermDebt','longTermDebt']], on='fiscalDateEnding', how='left')

In [124]:
kpis_df = kpis_df.fillna(0)

In [125]:
kpis_df.head()

,fiscalDateEnding,dividendPayoutCommonStock,dividendPayoutPreferredStock,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfPreferredStock,shortTermDebt,longTermDebt
0,2026-01-31,974000000.00,0.00,0.00,0.00,1371000000.00,7469000000.00
1,2025-01-31,834000000.00,0.00,0.00,0.00,288000000.00,8463000000.00
2,2024-01-31,395000000.00,0.00,0.00,0.00,1478000000.00,8459000000.00
3,2023-01-31,398000000.00,0.00,0.00,0.00,1250000000.00,9703000000.00
4,2022-01-31,399000000.00,0.00,0.00,0.00,144000000.00,10946000000.00


In [126]:
# Convert fiscalDateEnding to datetime for merging
kpis_df['fiscalDateEnding'] = pd.to_datetime(kpis_df['fiscalDateEnding'], errors='coerce')
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'], errors='coerce')

# Merge kpis_df with fcf_df on fiscalDateEnding
kpis_df = pd.merge(kpis_df, fcf_df[['fiscalDateEnding','growthCapex','freeCashFlow']], on='fiscalDateEnding', how='left')

In [127]:
kpis_df['dividendPayout'] = kpis_df['dividendPayoutCommonStock'] + kpis_df['dividendPayoutPreferredStock']
kpis_df['paymentsForRepurchase'] = kpis_df['paymentsForRepurchaseOfCommonStock'] + kpis_df['paymentsForRepurchaseOfPreferredStock']
kpis_df['debt'] = kpis_df['shortTermDebt'] + kpis_df['longTermDebt']

In [128]:
kpis_df.head()

,fiscalDateEnding,dividendPayoutCommonStock,dividendPayoutPreferredStock,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfPreferredStock,shortTermDebt,longTermDebt,growthCapex,freeCashFlow,dividendPayout,paymentsForRepurchase,debt
0,2026-01-31,974000000.00,0.00,0.00,0.00,1371000000.00,7469000000.00,3199000000,85825000000.00,974000000.00,0.00,8840000000.00
1,2025-01-31,834000000.00,0.00,0.00,0.00,288000000.00,8463000000.00,1372000000,55821000000.00,834000000.00,0.00,8751000000.00
2,2024-01-31,395000000.00,0.00,0.00,0.00,1478000000.00,8459000000.00,0,22459000000.00,395000000.00,0.00,9937000000.00
3,2023-01-31,398000000.00,0.00,0.00,0.00,1250000000.00,9703000000.00,289000000,2188000000.00,398000000.00,0.00,10953000000.00
4,2022-01-31,399000000.00,0.00,0.00,0.00,144000000.00,10946000000.00,0,6940000000.00,399000000.00,0.00,11090000000.00


In [129]:
kpis_df['debtRepayment'] = kpis_df['debt'].diff(-1) * -1
kpis_df.loc[kpis_df['debtRepayment'] < 0, 'debtRepayment'] = 0
kpis_df['debtRepayment'] = kpis_df['debtRepayment'].fillna(0)

In [130]:
kpis_df.head()

,fiscalDateEnding,dividendPayoutCommonStock,dividendPayoutPreferredStock,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfPreferredStock,shortTermDebt,longTermDebt,growthCapex,freeCashFlow,dividendPayout,paymentsForRepurchase,debt,debtRepayment
0,2026-01-31,974000000.00,0.00,0.00,0.00,1371000000.00,7469000000.00,3199000000,85825000000.00,974000000.00,0.00,8840000000.00,0.00
1,2025-01-31,834000000.00,0.00,0.00,0.00,288000000.00,8463000000.00,1372000000,55821000000.00,834000000.00,0.00,8751000000.00,1186000000.00
2,2024-01-31,395000000.00,0.00,0.00,0.00,1478000000.00,8459000000.00,0,22459000000.00,395000000.00,0.00,9937000000.00,1016000000.00
3,2023-01-31,398000000.00,0.00,0.00,0.00,1250000000.00,9703000000.00,289000000,2188000000.00,398000000.00,0.00,10953000000.00,137000000.00
4,2022-01-31,399000000.00,0.00,0.00,0.00,144000000.00,10946000000.00,0,6940000000.00,399000000.00,0.00,11090000000.00,0.00


In [131]:
kpis_df['organicGrowth%'] = kpis_df['growthCapex']*100 / kpis_df['freeCashFlow']
kpis_df['dividendPayout%'] = kpis_df['dividendPayout']*100 / kpis_df['freeCashFlow']
kpis_df['buybacks%'] = kpis_df['paymentsForRepurchase']*100 / kpis_df['freeCashFlow']
kpis_df['debtAmortization%'] = kpis_df['debtRepayment']*100 / kpis_df['freeCashFlow']

In [132]:
kpis_df['total%'] = kpis_df['organicGrowth%']+kpis_df['dividendPayout%']+kpis_df['buybacks%']+kpis_df['debtAmortization%']

In [133]:
kpis_numeric_cols = kpis_df.select_dtypes(include='number').columns
kpis_df[kpis_numeric_cols] = kpis_df[kpis_numeric_cols].clip(lower=0)

In [134]:
kpis_df = kpis_df[['fiscalDateEnding','organicGrowth%','dividendPayout%','buybacks%','debtAmortization%','total%']].copy()

In [135]:
kpis_df

,fiscalDateEnding,organicGrowth%,dividendPayout%,buybacks%,debtAmortization%,total%
0,2026-01-31,3.73,1.13,0.00,0.00,4.86
1,2025-01-31,2.46,1.49,0.00,2.12,6.08
2,2024-01-31,0.00,1.76,0.00,4.52,6.28
3,2023-01-31,13.21,18.19,0.00,6.26,37.66
4,2022-01-31,0.00,5.75,0.00,0.00,5.75
5,2021-01-31,0.78,10.29,0.00,0.00,11.07
6,2020-01-31,2.41,8.70,0.00,0.00,11.11
7,2019-01-31,11.34,12.45,0.00,0.00,23.79
8,2018-01-31,14.86,12.86,0.00,30.54,58.26
9,2017-01-31,0.00,36.92,0.00,0.00,36.92


## Visualización

### Tablas

#### Income Statement (Estado de resultados)

In [136]:
# List of columns to display in millions
columns_in_millions_income = ['totalRevenue', 'ebitda', 'depreciationDepletionAndAmortization',
                       'operatingIncome', 'interestExpense', 'interestIncome', 'netInterest',
                       'incomeBeforeTax', 'incomeTaxExpense', 'netIncome',
                       'commonStockSharesOutstanding']

In [137]:
income_table = convert_columns_to_millions(income_df.copy(), columns_in_millions_income)

In [138]:
income_table['fiscalDateEnding'] = income_table['fiscalDateEnding'].dt.year

In [139]:
income_table.head()

,fiscalDateEnding,totalRevenue,totalRevenue_YoY_growth%,ebitda,ebitdaMargin%,ebitda_YoY_growth%,depreciationDepletionAndAmortization,operatingIncome,ebitMargin%,operatingIncome_YoY_growth%,interestExpense,interestIncome,netInterest,incomeBeforeTax,incomeTaxExpense,taxRate%,netIncome,netIncomeMargin%,netIncome_YoY_growth%,reportedEPS,reportedEPS_YoY_growth%,commonStockSharesOutstanding,commonStockSharesOutstanding_YoY_growth%
0,2026,215938.00,65.47,144552.00,66.94,67.82,2843.00,130387.00,60.38,60.08,259.00,2300.00,2041.00,141450.00,21383.00,15.12,120067.00,55.60,64.75,4.78,59.76,24432.00,-1.50
1,2025,130497.00,114.20,86137.00,66.01,142.07,1864.00,81453.00,62.42,147.04,247.00,1786.00,1539.00,84026.00,11146.00,13.26,72880.00,55.85,144.89,2.99,130.69,24804.00,-0.55
2,2024,60922.00,125.85,35583.00,58.41,494.44,1508.00,32972.00,54.12,680.59,257.00,866.00,609.00,33818.00,4058.00,12.00,29760.00,48.85,581.32,1.30,289.49,24940.00,-0.52
3,2023,26974.00,0.22,5986.00,22.19,-47.26,1544.00,4224.00,15.66,-57.93,262.00,267.00,5.00,4181.00,-187.00,-4.47,4368.00,16.19,-55.21,0.33,-25.08,25070.00,-1.10
4,2022,26914.00,61.40,11351.00,42.18,99.46,1174.00,10041.00,37.31,121.56,236.00,29.00,-207.00,9941.00,189.00,1.90,9752.00,36.23,125.12,0.44,77.45,25350.00,0.92


In [140]:
income_mapping = {'fiscalDateEnding':f'{ticker}',
                   'totalRevenue':'Ventas',
                   'totalRevenue_YoY_growth%':'Crecimiento ventas interanual%',
                   'ebitda':'EBITDA',
                   'ebitdaMargin%':'Márgen EBITDA%',
                   'ebitda_YoY_growth%':'Crecimiento EBITDA interanual%',
                   'depreciationDepletionAndAmortization':'Depreciación y Amortización',
                   'operatingIncome':'EBIT',
                   'ebitMargin%':'Márgen EBIT%',
                   'operatingIncome_YoY_growth%':'Crecimiento EBIT interanual%',
                   'interestExpense':'Gastos por intereses',
                   'interestIncome':'Ingresos por intereses',
                   'netInterest':'Interés neto',
                   'incomeBeforeTax':'Ingreso antes de impuestos',
                   'incomeTaxExpense':'Gasto en impuestos',
                   'taxRate%':'Tasa impositiva%',
                   'netIncome':'Ingresos netos',
                   'netIncomeMargin%':'Márgen neto%',
                   'netIncome_YoY_growth%':'Crecimiento neto interanual%',
                   'reportedEPS':'EPS',
                   'reportedEPS_YoY_growth%':'Crecimiento EPS interanual%',
                   'commonStockSharesOutstanding':'Acciones diluidas en circulación',
                   'commonStockSharesOutstanding_YoY_growth%':'Crecimiento acciones interanual%'
                  }

In [141]:
income_table = income_table.rename(columns=income_mapping)
income_table.head()

,NVDA,Ventas,Crecimiento ventas interanual%,EBITDA,Márgen EBITDA%,Crecimiento EBITDA interanual%,Depreciación y Amortización,EBIT,Márgen EBIT%,Crecimiento EBIT interanual%,Gastos por intereses,Ingresos por intereses,Interés neto,Ingreso antes de impuestos,Gasto en impuestos,Tasa impositiva%,Ingresos netos,Márgen neto%,Crecimiento neto interanual%,EPS,Crecimiento EPS interanual%,Acciones diluidas en circulación,Crecimiento acciones interanual%
0,2026,215938.00,65.47,144552.00,66.94,67.82,2843.00,130387.00,60.38,60.08,259.00,2300.00,2041.00,141450.00,21383.00,15.12,120067.00,55.60,64.75,4.78,59.76,24432.00,-1.50
1,2025,130497.00,114.20,86137.00,66.01,142.07,1864.00,81453.00,62.42,147.04,247.00,1786.00,1539.00,84026.00,11146.00,13.26,72880.00,55.85,144.89,2.99,130.69,24804.00,-0.55
2,2024,60922.00,125.85,35583.00,58.41,494.44,1508.00,32972.00,54.12,680.59,257.00,866.00,609.00,33818.00,4058.00,12.00,29760.00,48.85,581.32,1.30,289.49,24940.00,-0.52
3,2023,26974.00,0.22,5986.00,22.19,-47.26,1544.00,4224.00,15.66,-57.93,262.00,267.00,5.00,4181.00,-187.00,-4.47,4368.00,16.19,-55.21,0.33,-25.08,25070.00,-1.10
4,2022,26914.00,61.40,11351.00,42.18,99.46,1174.00,10041.00,37.31,121.56,236.00,29.00,-207.00,9941.00,189.00,1.90,9752.00,36.23,125.12,0.44,77.45,25350.00,0.92


In [142]:
# Sort by Ticker in ascending order before transposing
income_viz = income_table.sort_values(by=ticker).set_index(ticker).T

In [143]:
income_viz

NVDA,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Ventas,3068.77,4097.86,3424.86,3326.45,3543.31,3997.93,4280.16,4130.00,4682.00,5010.00,6910.00,9714.00,11716.00,10918.00,16675.00,26914.00,26974.00,60922.00,130497.00,215938.00
Crecimiento ventas interanual%,13.37,33.53,-16.42,-2.87,6.52,12.83,7.06,-3.51,13.37,7.01,37.92,40.58,20.61,-6.81,52.73,61.40,0.22,125.85,114.20,65.47
EBITDA,561.01,969.54,142.47,117.69,442.74,852.50,874.47,759.00,1021.00,987.00,2150.00,3456.00,4216.00,3403.00,5691.00,11351.00,5986.00,35583.00,86137.00,144552.00
Márgen EBITDA%,18.28,23.66,4.16,3.54,12.49,21.32,20.43,18.38,21.81,19.70,31.11,35.58,35.98,31.17,34.13,42.18,22.19,58.41,66.01,66.94
Crecimiento EBITDA interanual%,60.74,72.82,-85.30,-17.40,276.19,92.55,2.58,-13.20,34.52,-3.33,117.83,60.74,21.99,-19.28,67.23,99.46,-47.26,494.44,142.07,67.82
Depreciación y Amortización,107.56,133.19,185.02,196.66,186.99,204.21,226.24,239.00,220.00,197.00,187.00,199.00,262.00,381.00,1098.00,1174.00,1544.00,1508.00,1864.00,2843.00
EBIT,453.45,836.35,-70.70,-98.94,255.75,648.30,648.24,496.00,759.00,747.00,1934.00,3210.00,3804.00,2846.00,4532.00,10041.00,4224.00,32972.00,81453.00,130387.00
Márgen EBIT%,14.78,20.41,-2.06,-2.97,7.22,16.22,15.15,12.01,16.21,14.91,27.99,33.05,32.47,26.07,27.18,37.31,15.66,54.12,62.42,60.38
Crecimiento EBIT interanual%,59.24,84.44,-108.45,-39.95,358.47,153.49,-0.01,-23.49,53.02,-1.58,158.90,65.98,18.50,-25.18,59.24,121.56,-57.93,680.59,147.04,60.08
Gastos por intereses,0.02,0.00,0.41,3.32,3.13,3.09,3.29,10.00,46.00,47.00,58.00,61.00,58.00,52.00,184.00,236.00,262.00,257.00,247.00,259.00


In [144]:
# Data viz:
generated_figs = generate_financial_bar_charts(income_viz)
for fig in generated_figs:
    fig.show()

#### FCF

In [145]:
columns_in_millions_fcf = ['ebitda','capitalExpenditures',
       'maintenanceCapex', 'growthCapex','interestExpense',
       'incomeTaxExpense','inventory','currentNetReceivables', 'currentAccountsPayable',
       'deferredRevenue','workingCapital','changeInWC', 'freeCashFlow']

In [146]:
fcf_table = convert_columns_to_millions(fcf_df.copy(), columns_in_millions_fcf)

In [147]:
fcf_table['fiscalDateEnding'] = fcf_table['fiscalDateEnding'].dt.year

In [148]:
fcf_table.head()

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
0,2026,133230.00,6042.00,2843.00,3199.00,259.00,21383.00,21403.00,38466.00,9812.00,381.00,49676.00,22920.00,85825.00,53.75,3.51,56.09,1.32,23.00,39.75,64.42
1,2025,83317.00,3236.00,1864.00,1372.00,247.00,11146.00,10080.00,23065.00,6310.00,79.00,26756.00,14239.00,55821.00,148.55,2.25,149.91,1.43,20.50,42.78,67.00
2,2024,34480.00,1069.00,1069.00,0.00,257.00,4058.00,5282.00,9999.00,2699.00,65.00,12517.00,6637.00,22459.00,926.46,0.90,931.81,1.75,20.55,36.87,65.14
3,2023,5768.00,1833.00,1544.00,289.00,262.00,-187.00,5159.00,3827.00,1193.00,1913.00,5880.00,1961.00,2188.00,-68.47,0.09,-68.12,5.72,21.80,8.11,37.93
4,2022,11215.00,976.00,976.00,0.00,236.00,189.00,2605.00,4650.00,1783.00,1553.00,3919.00,2874.00,6940.00,80.78,0.27,79.14,3.63,14.56,25.79,61.88


In [149]:
fcf_mapping = {'fiscalDateEnding':f'{ticker}',
               'ebitda':'EBITDA',
               'capitalExpenditures':'Gastos de Capital',
               'maintenanceCapex':'Gastos de Mantenimieto',
               'growthCapex':'Gastos de Expansión',
               'interestExpense':'Gastos por Intereses',
               'incomeTaxExpense':'Gasto en Impuestos',
               'inventory':'Inventarios',
               'currentNetReceivables':'Cuentas por Cobrar',
               'currentAccountsPayable':'Cuentas por Pagar',
               'deferredRevenue':'Ingresos Diferidos',
               'workingCapital':'Capital de Trabajo',
               'changeInWC':'Cambio en Capital de Trabajo',
               'freeCashFlow':'Flujo de Caja',
               'freeCashFlow_YoY_growth%':'Crecimiento Flujo Caja interanual%',
               'freeCashFlowPerShare':'Flujo de Caja por acción',
               'freeCashFlowPerShare_YoY_growth%':'Crecimiento Flujo Caja por acción interanual%',
               'maintenanceCapexMargin%':'Costo de Mantenimieto%',
               'workingCapitalMargin%':'Costo de Capital de Trabajo%',
               'freeCashFlowMargin%':'Márgen de Flujo de Caja%',
               'cashConversion%':'Conversión a Efectivo%'
}

In [150]:
fcf_table = fcf_table.rename(columns=fcf_mapping)
fcf_table.head()

,NVDA,EBITDA,Gastos de Capital,Gastos de Mantenimieto,Gastos de Expansión,Gastos por Intereses,Gasto en Impuestos,Inventarios,Cuentas por Cobrar,Cuentas por Pagar,Ingresos Diferidos,Capital de Trabajo,Cambio en Capital de Trabajo,Flujo de Caja,Crecimiento Flujo Caja interanual%,Flujo de Caja por acción,Crecimiento Flujo Caja por acción interanual%,Costo de Mantenimieto%,Costo de Capital de Trabajo%,Márgen de Flujo de Caja%,Conversión a Efectivo%
0,2026,133230.00,6042.00,2843.00,3199.00,259.00,21383.00,21403.00,38466.00,9812.00,381.00,49676.00,22920.00,85825.00,53.75,3.51,56.09,1.32,23.00,39.75,64.42
1,2025,83317.00,3236.00,1864.00,1372.00,247.00,11146.00,10080.00,23065.00,6310.00,79.00,26756.00,14239.00,55821.00,148.55,2.25,149.91,1.43,20.50,42.78,67.00
2,2024,34480.00,1069.00,1069.00,0.00,257.00,4058.00,5282.00,9999.00,2699.00,65.00,12517.00,6637.00,22459.00,926.46,0.90,931.81,1.75,20.55,36.87,65.14
3,2023,5768.00,1833.00,1544.00,289.00,262.00,-187.00,5159.00,3827.00,1193.00,1913.00,5880.00,1961.00,2188.00,-68.47,0.09,-68.12,5.72,21.80,8.11,37.93
4,2022,11215.00,976.00,976.00,0.00,236.00,189.00,2605.00,4650.00,1783.00,1553.00,3919.00,2874.00,6940.00,80.78,0.27,79.14,3.63,14.56,25.79,61.88


In [151]:
# Sort by Ticker in ascending order before transposing
fcf_viz = fcf_table.sort_values(by=ticker).set_index(ticker).T

In [152]:
fcf_viz

NVDA,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
EBITDA,561.01,969.54,114.32,97.72,442.74,852.50,874.47,735.00,979.00,944.00,2121.00,3409.00,4066.00,3227.00,5630.00,11215.00,5768.00,34480.00,83317.00,133230.00
Gastos de Capital,145.26,187.75,407.67,77.60,97.89,138.74,183.31,255.00,122.00,86.00,176.00,593.00,600.00,489.00,1128.00,976.00,1833.00,1069.00,3236.00,6042.00
Gastos de Mantenimieto,107.56,133.19,185.02,77.60,97.89,138.74,183.31,239.00,122.00,86.00,176.00,199.00,262.00,381.00,1098.00,976.00,1544.00,1069.00,1864.00,2843.00
Gastos de Expansión,37.69,54.55,222.65,0.00,0.00,0.00,0.00,16.00,0.00,0.00,0.00,394.00,338.00,108.00,30.00,0.00,289.00,0.00,1372.00,3199.00
Gastos por Intereses,0.02,0.00,0.41,3.32,3.13,3.09,3.29,10.00,46.00,47.00,58.00,61.00,58.00,52.00,184.00,236.00,262.00,257.00,247.00,259.00
Gasto en Impuestos,46.35,103.70,-12.91,-14.31,18.02,82.31,99.50,70.00,124.00,129.00,239.00,149.00,-245.00,174.00,77.00,189.00,-187.00,4058.00,11146.00,21383.00
Inventarios,354.68,358.52,537.83,330.67,345.52,340.30,419.69,387.76,482.89,418.00,794.00,796.00,1575.00,979.00,1826.00,2605.00,5159.00,5282.00,10080.00,21403.00
Cuentas por Cobrar,518.68,666.49,318.44,374.96,348.77,336.14,454.25,426.36,473.64,505.00,826.00,1265.00,1424.00,1657.00,2429.00,4650.00,3827.00,9999.00,23065.00,38466.00
Cuentas por Pagar,272.07,492.10,218.86,344.53,286.14,335.07,356.43,324.39,293.22,296.00,485.00,596.00,511.00,687.00,1201.00,1783.00,1193.00,2699.00,6310.00,9812.00
Ingresos Diferidos,0.00,0.00,0.00,111.95,347.71,455.81,589.32,475.12,488.93,453.00,20.00,2.00,14.00,1336.00,2009.00,1553.00,1913.00,65.00,79.00,381.00


In [153]:
# Data viz:
generated_figs = generate_financial_bar_charts(fcf_viz)
for fig in generated_figs:
    fig.show()

#### ROIC

In [154]:
columns_in_millions_roic = ['cashAndShortTermInvestments','shortTermInvestments','shortTermDebt','longTermDebt',
                            'capitalLeaseObligations','totalShareholderEquity','investedCapital']

In [155]:
roic_table = convert_columns_to_millions(roic_df.copy(), columns_in_millions_roic)

In [156]:
roic_table['fiscalDateEnding'] = roic_table['fiscalDateEnding'].dt.year

In [157]:
roic_table.head()

,fiscalDateEnding,cashAndShortTermInvestments,shortTermInvestments,shortTermDebt,longTermDebt,capitalLeaseObligations,totalShareholderEquity,investedCapital,ROA%,ROE%,ROIC%,reinvestmentRate%
0,2026,10605.00,51951.00,1371.00,7469.00,2572.00,157293.00,116754.00,58.06,76.33,94.79,3.73
1,2025,8589.00,34621.00,288.00,8463.00,1807.00,79327.00,55264.00,65.30,91.87,127.84,2.46
2,2024,7280.00,18704.00,1478.00,8459.00,1347.00,42978.00,35558.00,45.28,69.24,81.60,0.00
3,2023,3389.00,9907.00,1250.00,9703.00,902.00,22101.00,24049.00,10.61,19.76,18.35,13.21
4,2022,1990.00,19218.00,144.00,10946.00,741.00,26612.00,19225.00,22.07,36.65,51.24,0.00


In [158]:
roic_mapping = {'fiscalDateEnding':f'{ticker}',
                  'cashAndShortTermInvestments':'Efectivo',
                  'shortTermInvestments':'Inversiones a Corto Plazo',
                  'shortTermDebt':'Deuda a Corto Plazo',
                  'longTermDebt':'Deuda a Largo Plazo',
                  'capitalLeaseObligations':'Obligaciones de Arrendamiento',
                  'totalShareholderEquity':'Capital de los Accionistas',
                  'investedCapital':'Capital invertido',
                  'ROA%':'Retorno sobre los Activos%',
                  'ROE%':'Retorno sobre el Capital%',
                  'ROIC%':'Retorno sobre el Capital Invertido%',
                  'reinvestmentRate%':'Tasa de Reinversión%'
}

In [159]:
roic_table = roic_table.rename(columns=roic_mapping)

In [160]:
roic_table.head()

,NVDA,Efectivo,Inversiones a Corto Plazo,Deuda a Corto Plazo,Deuda a Largo Plazo,Obligaciones de Arrendamiento,Capital de los Accionistas,Capital invertido,Retorno sobre los Activos%,Retorno sobre el Capital%,Retorno sobre el Capital Invertido%,Tasa de Reinversión%
0,2026,10605.00,51951.00,1371.00,7469.00,2572.00,157293.00,116754.00,58.06,76.33,94.79,3.73
1,2025,8589.00,34621.00,288.00,8463.00,1807.00,79327.00,55264.00,65.30,91.87,127.84,2.46
2,2024,7280.00,18704.00,1478.00,8459.00,1347.00,42978.00,35558.00,45.28,69.24,81.60,0.00
3,2023,3389.00,9907.00,1250.00,9703.00,902.00,22101.00,24049.00,10.61,19.76,18.35,13.21
4,2022,1990.00,19218.00,144.00,10946.00,741.00,26612.00,19225.00,22.07,36.65,51.24,0.00


In [161]:
# Sort by Ticker in ascending order before transposing
roic_viz = roic_table.sort_values(by=ticker).set_index(ticker).T

In [162]:
roic_viz

NVDA,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Efectivo,544.41,726.97,417.69,447.22,665.36,667.88,732.79,1151.59,496.65,596.00,1766.00,4002.00,782.00,10896.00,847.00,1990.00,3389.00,7280.00,8589.00,10605.00
Inversiones a Corto Plazo,573.44,1082.51,837.70,1281.01,1825.20,2461.70,2995.10,3520.22,4126.69,4441.00,5032.00,3106.00,6640.00,1.00,10714.00,19218.00,9907.00,18704.00,34621.00,51951.00
Deuda a Corto Plazo,0.00,0.00,0.00,390.28,336.38,288.80,311.03,308.11,142.07,1500.00,827.00,15.00,91.00,91.00,999.00,144.00,1250.00,1478.00,288.00,1371.00
Deuda a Largo Plazo,0.00,0.00,25.63,24.45,23.39,21.44,19.00,1356.38,1398.43,87.00,1983.00,1985.00,1988.00,1991.00,5964.00,10946.00,9703.00,8459.00,8463.00,7469.00
Obligaciones de Arrendamiento,0.00,0.00,0.00,24.45,23.39,21.44,19.00,17.50,14.09,10.00,6.00,0.00,0.00,652.00,634.00,741.00,902.00,1347.00,1807.00,2572.00
Capital de los Accionistas,2006.92,2617.91,2394.65,2665.14,3181.46,4145.72,4827.70,4456.40,4417.98,4469.00,5762.00,7471.00,9342.00,12204.00,16893.00,26612.00,22101.00,42978.00,79327.00,157293.00
Capital invertido,1433.48,1535.40,1582.58,1823.31,1739.42,2015.70,2181.63,2618.16,1845.88,1625.00,3546.00,6365.00,4781.00,14937.00,13776.00,19225.00,24049.00,35558.00,55264.00,116754.00
Retorno sobre los Activos%,16.78,21.28,-0.90,-1.90,5.63,10.46,8.77,6.07,8.76,8.33,16.93,27.11,31.15,16.15,15.05,22.07,10.61,45.28,65.30,58.06
Retorno sobre el Capital%,22.36,30.47,-1.25,-2.55,7.96,14.02,11.65,9.87,14.28,13.74,28.91,40.78,44.33,22.91,25.64,36.65,19.76,69.24,91.87,76.33
Retorno sobre el Capital Invertido%,28.67,48.20,-3.12,-4.48,13.73,28.17,25.25,16.34,34.37,37.99,47.70,48.08,84.57,17.94,32.32,51.24,18.35,81.60,127.84,94.79


In [163]:
# Data viz:
generated_figs = generate_financial_bar_charts(roic_viz)
for fig in generated_figs:
    fig.show()

#### KPIs

In [164]:
kpis_table = kpis_df.copy()
kpis_table['fiscalDateEnding'] = kpis_table['fiscalDateEnding'].dt.year

In [165]:
kpis_mapping = {'fiscalDateEnding':f'{ticker}',
                  'organicGrowth%':'Crecimiento Organico%',
                  'dividendPayout%':'Dividendos Pagados%',
                  'buybacks%':'Recompras de Acciones%',
                  'debtAmortization%':'Amortización Neta de Deuda%',
                  'total%':'Total%'
}

In [166]:
kpis_table = kpis_table.rename(columns=kpis_mapping)

In [167]:
kpis_table.head()

,NVDA,Crecimiento Organico%,Dividendos Pagados%,Recompras de Acciones%,Amortización Neta de Deuda%,Total%
0,2026,3.73,1.13,0.00,0.00,4.86
1,2025,2.46,1.49,0.00,2.12,6.08
2,2024,0.00,1.76,0.00,4.52,6.28
3,2023,13.21,18.19,0.00,6.26,37.66
4,2022,0.00,5.75,0.00,0.00,5.75


In [168]:
# Sort by Ticker in ascending order before transposing
kpis_viz = kpis_table.sort_values(by=ticker).set_index(ticker).T

In [169]:
kpis_viz

NVDA,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Crecimiento Organico%,9.26,6.81,0.00,0.00,0.00,0.00,0.00,4.85,0.00,0.00,0.00,14.86,11.34,2.41,0.78,0.00,13.21,0.00,2.46,3.73
Dividendos Pagados%,0.00,0.00,-0.00,0.00,0.00,0.00,8.59,54.92,35.28,31.21,36.92,12.86,12.45,8.70,10.29,5.75,18.19,1.76,1.49,1.13
Recompras de Acciones%,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Amortización Neta de Deuda%,0.00,-0.00,-0.00,0.00,10.73,6.17,0.00,0.00,23.52,0.00,0.00,30.54,0.00,0.00,0.00,0.00,6.26,4.52,2.12,0.00
Total%,9.26,6.81,0.00,0.00,10.73,6.17,8.59,59.77,58.80,31.21,36.92,58.26,23.79,11.11,11.07,5.75,37.66,6.28,6.08,4.86


In [170]:
import plotly.graph_objects as go

# Exclude the 'Total%' row
kpis_viz_filtered = kpis_viz.drop('Total%')

# Clip values exceeding 100 to 100
kpis_viz_clipped = kpis_viz_filtered.clip(upper=100)

# Define a list of distinct colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

fig = go.Figure(data=[
    go.Bar(name='Crecimiento Organico%', x=kpis_viz_clipped.columns, y=kpis_viz_clipped.loc['Crecimiento Organico%'], marker_color=colors[0]),
    go.Bar(name='Dividendos Pagados%', x=kpis_viz_clipped.columns, y=kpis_viz_clipped.loc['Dividendos Pagados%'], marker_color=colors[1]),
    go.Bar(name='Recompras de Acciones%', x=kpis_viz_clipped.columns, y=kpis_viz_clipped.loc['Recompras de Acciones%'], marker_color=colors[2]),
    go.Bar(name='Amortización Neta de Deuda%', x=kpis_viz_clipped.columns, y=kpis_viz_clipped.loc['Amortización Neta de Deuda%'], marker_color=colors[3])
])

# Change the bar mode to stack
fig.update_layout(barmode='stack',
                  title='Distribución de KPIs a lo largo del tiempo',
                  xaxis_title='Año',
                  yaxis_title='Porcentaje',
                  yaxis=dict(range=[0, 100]), # Limit y-axis to 100%
                  xaxis=dict(
                      tickmode='array',
                      tickvals=kpis_viz_clipped.columns,
                      ticktext=kpis_viz_clipped.columns
                      )
                  )

fig.show()

#### Other graphs

In [171]:
# Perform a left join of income_table and fcf_table on the ticker column
financial_data = pd.merge(income_table, fcf_table, on=ticker, how='left')

In [172]:
# Perform a left join of income_table and roic_table on the ticker column
financial_data = pd.merge(financial_data, roic_table, on=ticker, how='left')

In [173]:
financial_data

,NVDA,Ventas,Crecimiento ventas interanual%,EBITDA_x,Márgen EBITDA%,Crecimiento EBITDA interanual%,Depreciación y Amortización,EBIT,Márgen EBIT%,Crecimiento EBIT interanual%,Gastos por intereses,Ingresos por intereses,Interés neto,Ingreso antes de impuestos,Gasto en impuestos,Tasa impositiva%,Ingresos netos,Márgen neto%,Crecimiento neto interanual%,EPS,Crecimiento EPS interanual%,Acciones diluidas en circulación,Crecimiento acciones interanual%,EBITDA_y,Gastos de Capital,Gastos de Mantenimieto,Gastos de Expansión,Gastos por Intereses,Gasto en Impuestos,Inventarios,Cuentas por Cobrar,Cuentas por Pagar,Ingresos Diferidos,Capital de Trabajo,Cambio en Capital de Trabajo,Flujo de Caja,Crecimiento Flujo Caja interanual%,Flujo de Caja por acción,Crecimiento Flujo Caja por acción interanual%,Costo de Mantenimieto%,Costo de Capital de Trabajo%,Márgen de Flujo de Caja%,Conversión a Efectivo%,Efectivo,Inversiones a Corto Plazo,Deuda a Corto Plazo,Deuda a Largo Plazo,Obligaciones de Arrendamiento,Capital de los Accionistas,Capital invertido,Retorno sobre los Activos%,Retorno sobre el Capital%,Retorno sobre el Capital Invertido%,Tasa de Reinversión%
0,2026,215938.00,65.47,144552.00,66.94,67.82,2843.00,130387.00,60.38,60.08,259.00,2300.00,2041.00,141450.00,21383.00,15.12,120067.00,55.60,64.75,4.78,59.76,24432.00,-1.50,133230.00,6042.00,2843.00,3199.00,259.00,21383.00,21403.00,38466.00,9812.00,381.00,49676.00,22920.00,85825.00,53.75,3.51,56.09,1.32,23.00,39.75,64.42,10605.00,51951.00,1371.00,7469.00,2572.00,157293.00,116754.00,58.06,76.33,94.79,3.73
1,2025,130497.00,114.20,86137.00,66.01,142.07,1864.00,81453.00,62.42,147.04,247.00,1786.00,1539.00,84026.00,11146.00,13.26,72880.00,55.85,144.89,2.99,130.69,24804.00,-0.55,83317.00,3236.00,1864.00,1372.00,247.00,11146.00,10080.00,23065.00,6310.00,79.00,26756.00,14239.00,55821.00,148.55,2.25,149.91,1.43,20.50,42.78,67.00,8589.00,34621.00,288.00,8463.00,1807.00,79327.00,55264.00,65.30,91.87,127.84,2.46
2,2024,60922.00,125.85,35583.00,58.41,494.44,1508.00,32972.00,54.12,680.59,257.00,866.00,609.00,33818.00,4058.00,12.00,29760.00,48.85,581.32,1.30,289.49,24940.00,-0.52,34480.00,1069.00,1069.00,0.00,257.00,4058.00,5282.00,9999.00,2699.00,65.00,12517.00,6637.00,22459.00,926.46,0.90,931.81,1.75,20.55,36.87,65.14,7280.00,18704.00,1478.00,8459.00,1347.00,42978.00,35558.00,45.28,69.24,81.60,0.00
3,2023,26974.00,0.22,5986.00,22.19,-47.26,1544.00,4224.00,15.66,-57.93,262.00,267.00,5.00,4181.00,-187.00,-4.47,4368.00,16.19,-55.21,0.33,-25.08,25070.00,-1.10,5768.00,1833.00,1544.00,289.00,262.00,-187.00,5159.00,3827.00,1193.00,1913.00,5880.00,1961.00,2188.00,-68.47,0.09,-68.12,5.72,21.80,8.11,37.93,3389.00,9907.00,1250.00,9703.00,902.00,22101.00,24049.00,10.61,19.76,18.35,13.21
4,2022,26914.00,61.40,11351.00,42.18,99.46,1174.00,10041.00,37.31,121.56,236.00,29.00,-207.00,9941.00,189.00,1.90,9752.00,36.23,125.12,0.44,77.45,25350.00,0.92,11215.00,976.00,976.00,0.00,236.00,189.00,2605.00,4650.00,1783.00,1553.00,3919.00,2874.00,6940.00,80.78,0.27,79.14,3.63,14.56,25.79,61.88,1990.00,19218.00,144.00,10946.00,741.00,26612.00,19225.00,22.07,36.65,51.24,0.00
5,2021,16675.00,52.73,5691.00,34.13,67.23,1098.00,4532.00,27.18,59.24,184.00,57.00,-127.00,4409.00,77.00,1.75,4332.00,25.98,54.94,0.25,72.40,25120.00,1.62,5630.00,1128.00,1098.00,30.00,184.00,77.00,1826.00,2429.00,1201.00,2009.00,1045.00,432.00,3839.00,-14.33,0.15,-15.69,6.58,6.27,23.02,68.19,847.00,10714.00,999.00,5964.00,634.00,16893.00,13776.00,15.05,25.64,32.32,0.78
6,2020,10918.00,-6.81,3403.00,31.17,-19.28,381.00,2846.00,26.07,-25.18,52.00,178.00,126.00,2970.00,174.00,5.86,2796.00,25.61,-32.48,0.15,-12.47,24720.00,-1.12,3227.00,489.00,381.00,108.00,52.00,174.00,979.00,1657.00,687.00,1336.00,613.00,-1861.00,4481.00,50.37,0.18,52.07,3.49,5.61,41.04,138.86,10896.00,1.00,91.00,1991.00,652.00,12204.00,14937.00,16.15,22.91,17.94,2.41
7,2019,11716.00,20.61,4216.00,35.98,21.99,262.00,3804.00,32.47,18.50,58.00,136.00,78.00,3896.00,-245.00,-6.29,4141.00,35.34,35.90,0.17,

In [174]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Crecimiento ventas interanual%'], mode='lines+markers', name='Crecimiento ventas interanual%'))
fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Crecimiento EPS interanual%'], mode='lines+markers', name='Crecimiento EPS interanual%'))
fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Crecimiento Flujo Caja por acción interanual%'], mode='lines+markers', name='Crecimiento Flujo Caja por acción interanual%'))

fig.update_layout(
    title='Crecimiento Histórico',
    xaxis_title='Año',
    yaxis_title='Crecimiento interanual (%)',
    xaxis=dict(
        tickmode='array',
        tickvals=financial_data[ticker],
        ticktext=financial_data[ticker]
    )
)

fig.show()

In [175]:
fig = go.Figure()

# Add the bar trace for 'Flujo de Caja'
fig.add_trace(go.Bar(x=financial_data[ticker], y=financial_data['Flujo de Caja'], name='Flujo de Caja'))

# Add the line trace for 'Retorno sobre el Capital Invertido%'
fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Retorno sobre el Capital Invertido%'], mode='lines+markers', name='Retorno sobre el Capital Invertido%', yaxis='y2'))

# Create axis objects
fig.update_layout(
    title='Flujo de Caja vs Retorno sobre el Capital Invertido%',
    xaxis_title='Año',
    yaxis=dict(
        title='Flujo de Caja (en millones)',
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue")
    ),
    yaxis2=dict(
        title='Retorno sobre el Capital Invertido (%)',
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying='y',
        side='right'
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=financial_data[ticker],
        ticktext=financial_data[ticker]
    )
)

fig.show()

In [176]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Márgen EBITDA%'], mode='lines+markers', name='Márgen EBITDA%'))
fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Márgen EBIT%'], mode='lines+markers', name='Márgen EBIT%'))
fig.add_trace(go.Scatter(x=financial_data[ticker], y=financial_data['Márgen de Flujo de Caja%'], mode='lines+markers', name='Márgen de Flujo de Caja%'))

fig.update_layout(
    title='Evolución histórica de los márgenes',
    xaxis_title='Año',
    yaxis_title='Margen (%)',
    xaxis=dict(
        tickmode='array',
        tickvals=financial_data[ticker],
        ticktext=financial_data[ticker]
    )
)

fig.show()

In [177]:
import plotly.express as px

# Filter columns to exclude those ending with '%'
filtered_columns = [col for col in financial_data.columns if not col.endswith('%')]

# Calculate the correlation of 'Flujo de Caja' with the filtered variables
# Ensure 'Flujo de Caja' is included if it doesn't end with '%'
if 'Flujo de Caja' not in filtered_columns:
    filtered_columns.append('Flujo de Caja')

# Select only the filtered columns for correlation calculation
financial_data_filtered = financial_data[filtered_columns]

correlations = financial_data_filtered.corr(method='kendall', numeric_only=True)['Flujo de Caja'].sort_values(ascending=False)

# Remove the correlation of 'Flujo de Caja' with itself
correlations = correlations.drop('Flujo de Caja')

# Create a bar chart using Plotly
fig = px.bar(correlations,
             x=correlations.index,
             y=correlations.values,
             title='Correlación de Flujo de Caja con otras variables financieras (sin porcentajes)')

fig.update_layout(xaxis_title='Variable',
                  yaxis_title='Coeficiente de Correlación')

fig.show()

## Exports

In [178]:
dataframes_to_export = {
    'income_df': income_df,
    'fcf_df': fcf_df,
    'roic_df': roic_df,
    'kpis_df': kpis_df,
    'income_table': income_table,
    'fcf_table': fcf_table,
    'roic_table': roic_table,
    'kpis_table': kpis_table,
    'income_viz': income_viz,
    'fcf_viz': fcf_viz,
    'roic_viz': roic_viz,
    'kpis_viz': kpis_viz
}

for name, df in dataframes_to_export.items():
    df.to_csv(f'{name}.csv', index=False)
    print(f'DataFrame {name} exportado a {name}.csv')


DataFrame income_df exportado a income_df.csv
DataFrame fcf_df exportado a fcf_df.csv
DataFrame roic_df exportado a roic_df.csv
DataFrame kpis_df exportado a kpis_df.csv
DataFrame income_table exportado a income_table.csv
DataFrame fcf_table exportado a fcf_table.csv
DataFrame roic_table exportado a roic_table.csv
DataFrame kpis_table exportado a kpis_table.csv
DataFrame income_viz exportado a income_viz.csv
DataFrame fcf_viz exportado a fcf_viz.csv
DataFrame roic_viz exportado a roic_viz.csv
DataFrame kpis_viz exportado a kpis_viz.csv
